# 8. XGBoost and neural model variants

The 68-variant factorial is the sole Paper 1 machine-learning workflow. The currently running execution record remains in `06C5_ranker_factorial.ipynb` until its launcher finishes; do not rename or edit that notebook during the run. Superseded standalone 06C notebooks have been removed.

This notebook is the post-run finalizer. It first displays experiment completion, then freezes one winner for each feature set × model-family cell (metadata and metadata + text crossed with XGBoost and neural network) per requested scope using development CV only. It refuses to run the freezer while `scripts/run_ranker_factorial.py` is active. `all` is the default; `root` is an appendix option.

In [ ]:
from pathlib import Path
import os

import pandas as pd

from commentgap_analysis.factorial_winners import (
    active_factorial_runner_processes,
    freeze_development_cv_winners,
)

scope_text = os.getenv("COMMENTGAP_MODEL_SCOPES", "all")
SCOPES = tuple(dict.fromkeys(part.strip() for part in scope_text.split(",") if part.strip()))
if not SCOPES or not set(SCOPES).issubset({"all", "root"}):
    raise ValueError(f"Invalid COMMENTGAP_MODEL_SCOPES={scope_text!r}")
FACTORIAL_ROOT = Path(os.getenv("COMMENTGAP_FACTORIAL_ROOT", "model_output/selection_2025/factorial_rankers"))
WINNER_ROOT = Path(os.getenv("COMMENTGAP_FACTORIAL_WINNER_ROOT", "model_output/selection_2025/paper1/factorial_winners"))
REPORT_DRAW_POLICIES = ("draw1",)
active_factorial_runner_processes()

In [ ]:
experiment = pd.read_csv(FACTORIAL_ROOT / "experiment_variants.csv")
completion = (
    experiment[experiment["scope"].isin(SCOPES)]
    .groupby(["family", "scope", "status"], as_index=False)
    .size()
)
display(completion)
completion

## Freeze development-CV winners

Run this cell only after the launcher above is absent. This reporting freeze is restricted to `draw1`: `mean10` averages ten deterministic audience tie draws, whereas `draw1` uses one, so restricting the reported set keeps the model-selection and CV scoring contract consistent across cells. The complete factorial remains available in the raw results for sensitivity analysis. Eligibility requires every selected planned variant/scope to be marked complete and to have exactly folds 0–4. Ranking is descending mean macro nDCG@k, lower fold SD, higher minimum-fold macro nDCG@k, then stable variant ID. No held-out artifact is read.

In [ ]:
winner_manifest = freeze_development_cv_winners(
    development_cv_path=FACTORIAL_ROOT / "development_cv_results.csv",
    experiment_variants_path=FACTORIAL_ROOT / "experiment_variants.csv",
    output_root=WINNER_ROOT,
    draw_policies=REPORT_DRAW_POLICIES,
    scopes=SCOPES,
)
winners = pd.read_csv(WINNER_ROOT / "development_cv_winners.csv")
ranking = pd.read_csv(WINNER_ROOT / "development_cv_variant_ranking.csv")
feature_set_labels = {"metadata": "metadata", "metadata_bge": "metadata + text"}
winner_keys = ["family", "scope", "feature_set"]
expected_winner_keys = {(family, scope, feature_set) for family in ("xgboost", "neural") for scope in SCOPES for feature_set in feature_set_labels}
observed_winner_keys = set(winners[winner_keys].itertuples(index=False, name=None))
if observed_winner_keys != expected_winner_keys or winners.duplicated(winner_keys).any():
    raise RuntimeError(f"Expected exactly one winner for each metadata/text × XGBoost/neural cell; expected={expected_winner_keys}, observed={observed_winner_keys}")
winner_table = (
    winners.assign(
        model_family=winners["family"].map({"xgboost": "XGBoost", "neural": "Neural network"}),
        feature_set_label=winners["feature_set"].map(feature_set_labels),
    )
    .sort_values(["scope", "feature_set", "family"])
)
display(winner_table[["scope", "feature_set_label", "model_family", "variant_id", "mean_macro_ndcg_at_k", "sd_macro_ndcg_at_k", "min_fold_macro_ndcg_at_k"]])
display(ranking.sort_values(["family", "scope", "feature_set", "development_cv_rank"]).groupby(["family", "scope", "feature_set"], group_keys=False).head(10))
winner_manifest